# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [20]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder
from preprocessing.metrics.reading.words import (
    all_tokens_from_aois,
)

import polars as pl
import contextlib

In [ ]:
from preprocessing.io.load import load_scanpaths

In [21]:
settings.RECALCULATE

False

In [ ]:
# If you have a specific config file, load it here:
settings.load_from_yaml(
    "/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/multipleye_settings_preprocessing.yaml"
)

In [22]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

Active Data Collection: MultiplEYE_SV_CH_Zurich_1_2026
Dataset Directory: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026


### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [23]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Recalculate:     {settings.RECALCULATE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

Include pilots:  True
Included:        ['009_SV_CH_1_ET1']
Excluded:        []
Output dir:      /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026
Run preflight:   True
Recalculate:     False


## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [24]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

2026-08-17 14:03:32,121 - preprocessing - INFO - Copying stimulus assets to /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026...


Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [105]:
settings.INCLUDE_SESSIONS = ['009_SV_CH_1_ET1']

In [106]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

2026-08-17 15:42:03,449 - preprocessing - INFO - Lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1_2026.py
2026-08-17 15:42:03,452 - preprocessing - INFO - JSON lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/MultiplEYE_SV_CH_Zurich_1_2026_lab_configuration.json
2026-08-17 15:42:03,454 - preprocessing - INFO - MultipleyeDataCollection initialized. data_root: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026/eye-tracking-sessions
2026-08-17 15:42:03,456 - preprocessing - INFO - Main config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1.py
2026

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [107]:
preprocessing.run_preflight_check(multipleye)

2026-08-17 15:42:06,228 - preprocessing - INFO - 
  Preflight check — all input files found


## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [108]:
multipleye.convert_edf_to_asc()

2026-08-17 15:42:08,733 - preprocessing - INFO - Starting EDF to ASC conversion for 1 sessions.
Converting EDF to ASC: 100%|██████████| 1/1 [00:00<00:00, 828.42it/s]
2026-08-17 15:42:08,744 - preprocessing - INFO - EDF to ASC conversion completed.


Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [109]:
multipleye.sessions

{'009_SV_CH_1_ET1': Session(participant_id=9, session_identifier='009_SV_CH_1_ET1', is_pilot=True, session_folder_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026/eye-tracking-sessions/pilot_sessions/009_SV_CH_1_ET1'), session_file_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026/eye-tracking-sessions/pilot_sessions/009_SV_CH_1_ET1/009svch1.edf'), session_file_name='009svch1.edf', asc_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/asc/009_SV_CH_1_ET1/009_SV_CH_1_ET1.asc'), stimuli='unknown', randomization_version='unknown', stimulus_folder_name='unknown', completed_stimuli_ids='unknown', completed_stimuli_names='unknown', question_order='unknown', stimulus_order_ids='unknown', messages='unknown', stimuli_trial_mapping='unknown', stimulus_start_end_ts='unknown', logfile='unknown', interrupted='unkn

In [110]:
multipleye.prepare_session_level_information()

Preparing session 009_SV_CH_1_ET1: 100%|██████████| 1/1 [00:11<00:00, 11.10s/it]


## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [111]:
# pick only one session as an example to work with in the next steps
sessions = list(multipleye)  # list of Session objects
sess = sessions[0]  # a real Session
sid = sess.sid  # get the session ID (Sid) from the Session
sid

Sid(pid='009', lang='SV', country='CH', lab='1', session='ET1', session_id=1, postfix='')

In [ ]:
type(sess)

Checking if the number of expected files is correct

In [ ]:
asc = sess.asc_path

# check whether the raw data was calculated before and whether it is complete
raw_data_folder = sess.sid.raw_data_dir
num_expected_files = len(sess.completed_stimuli_ids)
try:
    num_files = len(list(raw_data_folder.glob("*.csv")))
except FileNotFoundError:
    num_files = 0

In [ ]:
num_files

In [ ]:
settings.RECALCULATE = False

In [ ]:
settings.RECALCULATE

In [ ]:
num_expected_files

In [ ]:
num_files

In [ ]:
if num_expected_files == num_files and not settings.RECALCULATE:
    # Loading previously extracted raw data
    print("just loading")
    gaze = preprocessing.load_trial_level_raw_data(
        sess.sid,
        trial_columns=settings.TRIAL_COLS,
        load_metadata=True,
    )

else:
    # Extract raw data from asc file
    print("recalculating")
    gaze = preprocessing.load_gaze_data(
        asc_file=asc,
        lab_config=sess.lab_config,
        sid=sess.sid,
        trial_cols=settings.TRIAL_COLS,
        messages=settings.ANSWER_MSG_PATTERNS,
    )

    gaze.samples = gaze.samples.filter(
        pl.col("stimulus").is_in(sess.completed_stimuli_names)
    )

    preprocessing.save_raw_data(sess.sid, gaze)
    preprocessing.save_session_metadata(sess.sid, gaze)

### Load previously processed file

In [73]:
gaze = preprocessing.load_trial_level_raw_data(
    sess.sid,
    trial_columns=settings.TRIAL_COLS,
    load_metadata=True,
)

In [74]:
gaze

time,position_x,position_y,velocity_x,velocity_y,pupil,page,trial,stimulus,pixel
i64,f64,f64,f64,f64,f64,str,str,str,list[f64]
4155117,-15.702869,-10.555472,-0.22409,0.76424,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]"
4155118,-15.695336,-10.558051,-0.16456,0.792012,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]"
4155119,-15.695336,-10.581259,-0.044335,0.746959,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]"
4155120,-15.682779,-10.542577,-0.059119,0.684403,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]"
4155121,-15.68529,-10.583838,-0.118434,0.704238,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]"
…,…,…,…,…,…,…,…,…,…
2406444,15.151475,11.232203,0.378963,-3.143524,542.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1224.1, 926.9]"
2406445,15.126228,11.288671,0.559596,-3.25244,547.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.1, 929.1]"
2406446,15.138852,11.245039,0.845663,-3.313736,550.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.6, 927.4]"


In [34]:
gaze.samples

time,pupil,page,trial,stimulus,pixel
i64,f64,str,str,str,list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]"
…,…,…,…,…,…
2406444,542.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1224.1, 926.9]"
2406445,547.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.1, 929.1]"
2406446,550.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.6, 927.4]"


### Output directory structure

All preprocessed data is organised by data type under `preprocessed_data/<data_collection_name>/`. Each data type folder contains one subfolder per session:

```
preprocessed_data/<dcn>/
├── raw_data/
│   └── <session_save_name>/
├── fixations/
│   └── <session_save_name>/
├── saccades/
│   └── <session_save_name>/
├── scanpaths/
│   └── <session_save_name>/
├── reading_measures/
│   └── <session_save_name>/
├── sanity_checks/
│   └── <session_save_name>/
├── metadata/
│   └── <session_save_name>/
│       ├── gaze_metadata.json
│       ├── experiment.yaml
│       ├── calibrations.tsv
│       ├── calibrations.feather
│       ├── validations.tsv
│       ├── validations.feather
│       └── <session_idf>_overview.yaml
├── participant_data.csv
├── <dcn>_overview.yaml
└── stimuli_<dcn>/
```

The `Sid` object provides convenient properties to access each path:

In [35]:
print(f"Raw data:        {sid.raw_data_dir}")
print(f"Metadata:        {sid.metadata_dir}")
print(f"Fixations:       {sid.fixations_dir}")
print(f"Saccades:        {sid.saccades_dir}")
print(f"Scanpaths:       {sid.scanpaths_dir}")
print(f"Reading measures: {sid.reading_measures_dir}")

Raw data:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/raw_data/009_SV_CH_1_ET1
Metadata:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/metadata/009_SV_CH_1_ET1
Fixations:       /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/fixations/009_SV_CH_1_ET1
Saccades:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/saccades/009_SV_CH_1_ET1
Scanpaths:       /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/scanpaths/009_SV_CH_1_ET1
Reading measures: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/reading_measures/009_SV_CH_1_ET1


In order to have the metadata which is extracted by pymovements available to create out session overview, we get this information from pymovements and store it in our session object.

In [36]:
sess.pm_gaze_metadata = gaze._metadata
sess.calibrations = gaze.calibrations
sess.validations = gaze.validations

### Coordinate and Velocity Preprocessing

Eye movements are recorded in screen pixel coordinates, which depend on stimulus size and monitor setup. To compare gaze behavior across participants, screens, or datasets, it is standard to convert pixel positions 
into **degrees of visual angle (dva)**. Next, we compute **gaze velocity**, which allows us to detect saccades and distinguish them from fixations.

In [37]:
# inspect the gaze samples
gaze.samples.head()

time,pupil,page,trial,stimulus,pixel
i64,f64,str,str,str,list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]"


In [38]:
import copy

In [39]:
no_prepro_gaze = copy.deepcopy(gaze)

In [40]:
no_prepro_gaze

time,pupil,page,trial,stimulus,pixel
i64,f64,str,str,str,list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]"
…,…,…,…,…,…
2406444,542.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1224.1, 926.9]"
2406445,547.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.1, 929.1]"
2406446,550.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.6, 927.4]"


In [41]:
preprocessing.preprocess_gaze(gaze)

In [42]:
# inspect the preprocessed gaze samples, the dataframe should now also contain a position in dva and velocity columns
gaze.samples.head()

time,pupil,page,trial,stimulus,pixel,position,velocity
i64,f64,str,str,str,list[f64],list[f64],list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]","[-15.702869, -10.555472]","[-0.22409, 0.76424]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]","[-15.695336, -10.558051]","[-0.16456, 0.792012]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]","[-15.695336, -10.581259]","[-0.044335, 0.746959]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]","[-15.682779, -10.542577]","[-0.059119, 0.684403]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]","[-15.68529, -10.583838]","[-0.118434, 0.704238]"


In [51]:
import pymovements as pm
from preprocessing.models.sid import Sid
import contextlib

In [71]:
def save_prepro_raw_data(sid: Sid, data: pm.Gaze) -> None:
    """
    Saves raw gaze data in separate csv files per trial.

    Parameters
    ----------
    sid : Sid
        The session identifier.
    data : pm.Gaze
        The gaze data as a pymovements Gaze object.
    """
    directory = sid.raw_data_dir
    directory.mkdir(parents=True, exist_ok=True)

    new_data = data.clone()

    trials = new_data.split(by="trial", as_dict=False)

    for trial in trials:
        with contextlib.suppress(Warning):
            trial.unnest()
        df = trial.samples
        trial = df["trial"][0]
        stimulus = df["stimulus"][0]
        name = f"{str(sid)}_{trial}_{stimulus}_raw_data.csv"
        df = df["time", "pixel_x", "pixel_y","position_x", "position_y", "velocity_x", "velocity_y", "pupil", "page"]
        df.write_csv(directory / name)

In [80]:
import re
import json
import yaml

In [86]:
def load_prepro_raw_data(
    sid: Sid,
    trial_columns: list[str],
    file_pattern: str | None = None,
    load_metadata: bool = False,
) -> pm.Gaze:
    """Load trial-level raw data from multiple CSV files and construct a gaze object.

    This function aggregates raw data files containing gaze data for one or more trials.

    Parameters
    ----------
    sid : Sid
        The session identifier.
    trial_columns : list of str
        Column names that uniquely identify a trial within the data.
    file_pattern : str, optional
        The file search pattern for raw data CSV files. Defaults to None, which uses settings.RAW_DATA_FILE_GLOB.
    load_metadata : bool, optional
        Whether to load metadata files (`gaze_metadata.json`, `experiment.yaml`,
        `validations.tsv`, `calibrations.tsv`) to enrich the gaze object.

    Returns
    -------
    pm.Gaze
        A gaze object containing the trial-level aggregated gaze data along with
        any associated metadata, validations, calibrations, and experiment settings, if provided.
    """
    data_folder = sid.raw_data_dir
    if file_pattern is None:
        file_pattern = settings.RAW_DATA_FILE_GLOB

    regex_name = settings.RAW_DATA_FILENAME_REGEX

    initial_df = pl.DataFrame()

    for file in data_folder.glob(file_pattern):
        trial_df = pl.read_csv(
            file,
            schema_overrides={
                "time": pl.Float64,
                "pupil": pl.Float64,
                "pixel_x": pl.Float64,
                "pixel_y": pl.Float64,
                "position_x": pl.Float64,
                "position_y": pl.Float64,
                "velocity_x": pl.Float64,
                "velocity_y": pl.Float64,
                "page": pl.Utf8,
            },
        )
        match = re.match(regex_name, file.stem)
        trial_df = trial_df.with_columns(
            pl.lit(match.group("trial")).alias("trial"),
            pl.lit(match.group("stimulus")).alias("stimulus"),
        )

        initial_df = initial_df.vstack(trial_df)

    if initial_df.is_empty():
        raise ValueError(
            f"No raw data files found in {data_folder} with pattern {file_pattern}"
        )

    gaze = pm.Gaze(
        initial_df,
        trial_columns=trial_columns,
        pixel_columns=["pixel_x", "pixel_y"],
        position_columns=["position_x", "position_y"],
        velocity_columns=["velocity_x", "velocity_y"],
    )

    if load_metadata:
        metadata_path = sid.metadata_dir

        with open(metadata_path / "gaze_metadata.json", encoding="utf8") as f:
            metadata = json.load(f)

        gaze._metadata = metadata

        with open(metadata_path / "experiment.yaml") as f:
            exp = yaml.safe_load(f)

        with open(metadata_path / "validations.tsv", encoding="utf8") as f:
            validations_df = pl.read_csv(f, separator="\t")

        gaze.validations = validations_df

        with open(metadata_path / "calibrations.tsv", encoding="utf8") as f:
            calibrations_df = pl.read_csv(f, separator="\t")

        gaze.calibrations = calibrations_df

        exp = pm.Experiment.from_dict(exp)

        gaze.experiment = exp

    return gaze

In [112]:
raw_data_folder = sess.sid.raw_data_dir

In [113]:
one_file = list(raw_data_folder.glob("*.csv"))[0]

In [114]:
"position_x" in pl.read_csv(one_file).columns

True

In [98]:
gaze = load_prepro_raw_data(
    sess.sid,
    trial_columns=settings.TRIAL_COLS,
    load_metadata=True,
)

ColumnNotFoundError: column position_x from position_columns is not available in samples dataframe

In [89]:
sess.sid

Sid(pid='009', lang='SV', country='CH', lab='1', session='ET1', session_id=1, postfix='')

In [88]:
gaze.samples

time,pupil,page,trial,stimulus,pixel,position,velocity
i64,f64,str,str,str,list[f64],list[f64],list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]","[-15.702869, -10.555472]","[-0.22409, 0.76424]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]","[-15.695336, -10.558051]","[-0.16456, 0.792012]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]","[-15.695336, -10.581259]","[-0.044335, 0.746959]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]","[-15.682779, -10.542577]","[-0.059119, 0.684403]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]","[-15.68529, -10.583838]","[-0.118434, 0.704238]"
…,…,…,…,…,…,…,…
2406444,542.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1224.1, 926.9]","[15.151475, 11.232203]","[0.378963, -3.143524]"
2406445,547.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.1, 929.1]","[15.126228, 11.288671]","[0.559596, -3.25244]"
2406446,550.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.6, 927.4]","[15.138852, 11.245039]","[0.845663, -3.313736]"


In [84]:
gaze.samples

time,position_x,position_y,velocity_x,velocity_y,pupil,page,trial,stimulus,pixel
i64,f64,f64,f64,f64,f64,str,str,str,list[f64]
4155117,-15.702869,-10.555472,-0.22409,0.76424,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]"
4155118,-15.695336,-10.558051,-0.16456,0.792012,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]"
4155119,-15.695336,-10.581259,-0.044335,0.746959,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]"
4155120,-15.682779,-10.542577,-0.059119,0.684403,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]"
4155121,-15.68529,-10.583838,-0.118434,0.704238,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]"
…,…,…,…,…,…,…,…,…,…
2406444,15.151475,11.232203,0.378963,-3.143524,542.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1224.1, 926.9]"
2406445,15.126228,11.288671,0.559596,-3.25244,547.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.1, 929.1]"
2406446,15.138852,11.245039,0.845663,-3.313736,550.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.6, 927.4]"


In [59]:
directory = sid.raw_data_dir
directory.mkdir(parents=True, exist_ok=True)

In [60]:
new_data = gaze.clone()

In [61]:
trials = new_data.split(by="trial", as_dict=False)

In [64]:
trial = trials[0]

In [65]:
with contextlib.suppress(Warning):
    trial.unnest()
df = trial.samples

In [66]:
df

time,pupil,page,trial,stimulus,pixel_x,pixel_y,position_x,position_y,velocity_x,velocity_y
i64,f64,str,str,str,f64,f64,f64,f64,f64,f64
2170575,652.0,"""page_1""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",80.5,103.7,-15.111077,-10.470344,-0.544006,2.03784
2170576,653.0,"""page_1""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",81.3,103.2,-15.090872,-10.483245,-0.415798,1.847938
2170577,642.0,"""page_1""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",78.3,105.9,-15.166621,-10.413566,-0.439114,1.478056
2170578,645.0,"""page_1""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",80.5,106.8,-15.111077,-10.390333,-0.545122,1.40235
2170579,651.0,"""page_1""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",78.8,104.4,-15.154,-10.452281,-0.490519,0.990696
…,…,…,…,…,…,…,…,…,…,…
2309886,596.0,"""question_13131""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",1218.1,930.6,14.999901,11.327159,-0.057979,0.84019
2309887,594.0,"""question_13131""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",1216.2,936.3,14.951858,11.473319,-0.097112,0.853224
2309888,598.0,"""question_13131""","""PRACTICE_trial_1""","""Enc_WikiMoon_13""",1213.7,933.0,14.88861,11.388718,-0.245029,1.058165


In [72]:
save_prepro_raw_data(sid,gaze)

## Stage 2a: Detect Events and Compute Their Properties

Eye-tracking data are typically segmented into events, i.e. `fixations` and `saccades`. Fixations represent moments when the eyes remain relatively still, allowing visual information to be processed, while saccades are the rapid movements between fixations that reposition the gaze. Detecting these events and computing their properties, such as `dispersion`, fixation `duration`, saccade `amplitude`, and `peak velocity`, provides the foundation for analyzing visual behavior and understanding how participants explore a stimulus.

### Fixations

We can detect fixations by applying the `I-VT` or the `I-DT` method.

The **I-VT (Velocity-Threshold Identification)** method distinguishes fixation and saccade points based on their point-to-point velocities. Each point is classified as a fixation if its velocity is below the specified threshold. Consecutive fixation points are then merged into a single fixation. A threshold of 20 degrees/second is commonly used as a default maximum value. Read more about [the IVT algorithm in the documentation](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.ivt.html) 

The **I-DT (Dispersion-Threshold Identification)** method finds fixations by grouping consecutive points within a maximum separation (dispersion) threshold and a minimum duration threshold. The algorithm slides a moving window across the data: if the dispersion within the window is below the threshold, the window represents a fixation and is gradually expanded until the dispersion exceeds the threshold.
Read more about [our implementation of the IDT method](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.idt.html).

We use the `I-VT` algorithm with the following key deafault parameters:
- `minimum duration`: 100 ms 
- `velocity threshold`: 20.0

Such properties as `location`, containing the centroid coordinates of each fixation, and `dispersion` will also be calculated.

In [43]:
# create or load fixation data
fixation_data_folder = sess.sid.fixations_dir
saccade_data_folder = sess.sid.saccades_dir

In [ ]:
saccade_data_folder

In [44]:
settings.RUN_FIXATION_DETECTION = True
settings.RUN_SACCADE_DETECTION = True

In [ ]:
if settings.RUN_FIXATION_DETECTION or settings.RUN_SACCADE_DETECTION:
    if gaze is None:
        print(f"Gaze data missing for {sess.sid}. Skipping event detection.")
    else:
        num_expected_files = len(sess.completed_stimuli_ids)

        try:
            num_fix_files = len(list(fixation_data_folder.glob("*.csv")))
            num_sacc_files = len(list(saccade_data_folder.glob("*.csv")))
        except FileNotFoundError:
            num_fix_files = 0
            num_sacc_files = 0

        if (
            num_expected_files == num_fix_files
            and num_expected_files == num_sacc_files
            and not settings.RECALCULATE
        ):
            # Loading events if all files exist and the recalculate flag is not active
            print(f"Loading events {sess.sid}:")
            gaze = preprocessing.load_trial_level_events_data(
                gaze,
                sess.sid,
                event_type=settings.FIXATION,
                file_pattern=None,
            )

            gaze = preprocessing.load_trial_level_events_data(
                gaze,
                sess.sid,
                event_type=settings.SACCADE,
                file_pattern=None,
            )

        else:
            # If files were not complete or recalculation is active we run event detection
            print(f"Detecting events {sess.sid}:")

            if settings.RUN_FIXATION_DETECTION:
                print("Detecting fixations")
                preprocessing.detect_fixations(gaze)

                print("Saving Fixations")
                preprocessing.save_events_data(
                    settings.FIXATION,
                    sess.sid,
                    "trial",
                    ["trial", "stimulus"],
                    ["onset", "duration", "location_x", "location_y", "page"],
                    gaze,
                )

            if settings.RUN_SACCADE_DETECTION:
                print("Detecting Saccades")
                preprocessing.detect_saccades(gaze)

                print("Saving Saccades")
                preprocessing.save_events_data(
                    settings.SACCADE,
                    sess.sid,
                    "trial",
                    ["trial", "stimulus"],
                    [
                        "onset",
                        "duration",
                        "amplitude",
                        "peak_velocity",
                        "dispersion",
                        "page",
                    ],
                    gaze,
                )

            # Unnest event columns (e.g. location struct -> location_x/location_y)
            # so downstream code doesn't need to handle struct columns.
            if gaze is not None and gaze.events is not None:
                with contextlib.suppress(Warning):
                    gaze.events.unnest()

else:
    print(f"Skipping event detection {sess.sid}:")
    # Load existing if available
    if (
        gaze is not None
        and fixation_data_folder.exists()
        and saccade_data_folder.exists()
    ):
        print(f"Using existing event data for {sess.sid}")
        gaze = preprocessing.load_trial_level_events_data(
            gaze,
            sess.sid,
            event_type=settings.FIXATION,
            file_pattern=None,
        )
        gaze = preprocessing.load_trial_level_events_data(
            gaze,
            sess.sid,
            event_type=settings.SACCADE,
            file_pattern=None,
        )

In [45]:
after_prepro_gaze = copy.deepcopy(gaze)
after_prepro_gaze

time,pupil,page,trial,stimulus,pixel,position,velocity
i64,f64,str,str,str,list[f64],list[f64],list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]","[-15.702869, -10.555472]","[-0.22409, 0.76424]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]","[-15.695336, -10.558051]","[-0.16456, 0.792012]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]","[-15.695336, -10.581259]","[-0.044335, 0.746959]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]","[-15.682779, -10.542577]","[-0.059119, 0.684403]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]","[-15.68529, -10.583838]","[-0.118434, 0.704238]"
…,…,…,…,…,…,…,…
2406444,542.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1224.1, 926.9]","[15.151475, 11.232203]","[0.378963, -3.143524]"
2406445,547.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.1, 929.1]","[15.126228, 11.288671]","[0.559596, -3.25244]"
2406446,550.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.6, 927.4]","[15.138852, 11.245039]","[0.845663, -3.313736]"


In [46]:
gaze = preprocessing.load_trial_level_events_data(
    gaze,
    sess.sid,
    event_type=settings.FIXATION,
    file_pattern=None,
)

gaze = preprocessing.load_trial_level_events_data(
    gaze,
    sess.sid,
    event_type=settings.SACCADE,
    file_pattern=None,
)

In [49]:
no_prepro_gaze

time,pupil,page,trial,stimulus,pixel
i64,f64,str,str,str,list[f64]
4155117,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.0, 100.4]"
4155118,539.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 100.3]"
4155119,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.3, 99.4]"
4155120,535.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.8, 100.9]"
4155121,540.0,"""page_1""","""trial_4""","""Lit_BrokenApril_9""","[57.7, 99.3]"
…,…,…,…,…,…
2406444,542.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1224.1, 926.9]"
2406445,547.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.1, 929.1]"
2406446,550.0,"""question_7131""","""PRACTICE_trial_2""","""Lit_NorthWind_7""","[1223.6, 927.4]"


### Saccades

Saccades are rapid eye movements that shift the point of fixation from one location to another. We detect saccades (or micro-saccades) from the velocity sequence of gaze data using the [microsaccades algorithm](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.microsaccades.html#pymovements.events.detection.microsaccades). This algorithm implements a noise-adaptive velocity threshold, meaning that the detection threshold automatically scales with the noise level of the velocity signal. Such properties as `amplitude` and `peak velocity` of the detected saccades will also be calcuated.

The key default parameters are:
- `threshold_factor`: Multiplier used to determine the velocity threshold relative to the noise level of the signal. The default value is 6. A higher factor makes the algorithm more conservative (detects fewer saccades), while a lower factor makes it more sensitive.
- `minimum_duration`: Defines how long a velocity peak must persist to be classified as a saccade. The duration is expressed in the same units as timesteps. If no timesteps are provided, the value refers to the number of samples (default = 6), which corresponds to about 12 ms at a 500 Hz sampling rate. Shorter events are ignored as noise. 

Save our events data.

## Create new Load scanpaths function

In [ ]:
import copy

In [ ]:
unmapped_gaze = copy.deepcopy(gaze)

In [ ]:
preprocessing.map_fixations_to_aois(gaze, sess.stimuli)

In [ ]:
preprocessing.save_scanpaths(sid, gaze)

In [ ]:
unmapped_gaze.events.frame

In [ ]:
gaze.events.frame
# Missing columns: char_idx, char, top_left_x, top_left_y, width, height, char_idx_in_line, line_idx, word_idx, word_idx_in_line, word

Let's try and get all the scanpath files

In [ ]:
trial_col = "trial"
stimulus_col = "stimulus"

In [ ]:
scanpaths = load_scanpaths(gaze, sid)

In [ ]:
unmapped_gaze.events.frame

In [ ]:
scanpaths

In [ ]:
matched_events = unmapped_gaze.events.frame.join(
    scanpaths, on=("onset", "trial", "stimulus", "page", "name"), how="left"
)
matched_events = matched_events.drop(
    "duration_right", "location_x_right", "location_y_right"
)

In [ ]:
gaze.events.frame

In [ ]:
matched_events.equals(gaze.events.frame)

In [ ]:
matched_events.select(gaze.events.frame.columns).equals(gaze.events.frame)

Missing: offset, amplitude, peak velocity, dispersion
Scanpaths only includes fixations?

In [ ]:
matched_events

In [ ]:
unmapped_gaze

In [ ]:
output_gaze = copy.deepcopy(unmapped_gaze)
output_gaze.events.frame = matched_events
output_gaze

In [ ]:
output_gaze == gaze

In [ ]:
gaze

## Stage 2b: Map Fixations to AOIs

Once we have the fixations, we can map each of them to the AOIs of the stimulus. The resulting scanpath can then be saved. Note that this features is not yet completely finished.

In [ ]:
preprocessing.map_fixations_to_aois(gaze, sess.stimuli)

In [ ]:
# The resulting mapping can be stored as a scanpath, which is a sequence of AOIs that were fixated in the order they were fixated.
preprocessing.save_scanpaths(sid, gaze)

In [ ]:
# save metadata again
preprocessing.save_session_metadata(sid, gaze)

## Stage 3: Calculate AOI-based Measures

In this last step, we calculate the aoi-based measures. These are also refered to as reading measures, as they are typically used in reading research. They include measures such as first pass fixation duration (FPF), total fixation count (TFC), regression path duration (RPD), and many more. These measures are calculated based on the fixations that were mapped to the AOIs in the previous step.

### Fixation-based Metrics

As an intermediate step, the fixations are annotated. These annoataions include:
- The run ID. This ID specifies continuous sequences of fixations on the same word. It is used to calculate first pass and second pass measures.
- Whether the fixation is within the first pass or not
- The index of the preceding word and the following word
- If the saccade entering or leaving the fixation is a regression or not
- Whether it is the first fixation on the word or not

This information is necessary to calculate the reading measures in the next step.

In [ ]:
def compute_reading_measures(
    fixations: pl.DataFrame,
    aois: pl.DataFrame,
    *,
    word_index_column: str = "word_idx",
    word_column: str = "word",
    char_index_column: str = "char_idx",
) -> pl.DataFrame:
    """Compute reading measures from fixation sequences.

    This function expects fixations annotated with AOI data. See
    :py:meth:`~pymovements.Events.map_to_aois` for further details.

    Parameters
    ----------
    fixations : pl.DataFrame
        DataFrame with fixation data, containing the column specified by ``word_index_column`` and by ``char_index_column``.
    aois : pl.DataFrame
        DataFrame with AOI data, containing the columns specified by ``word_index_column`` and
        ``word_column``.
    word_index_column : str
        Shared column name in ``fixations`` and ``aois`` that corresponds to the word index of the
        text.
        (default: ``'aoi'``)
    word_column : str
        Column in ``aois`` with the content within each AOI.
        (default: ``'word'``)
    char_index_column : str
        Column in ``fixations`` that corresponds to the character index of the text.

    Returns
    -------
    pl.DataFrame
        DataFrame with computed reading measures.
    """
    # Append an extra dummy fixation to have the next fixation for the actual last fixation.
    dummy_fixation_dict: dict[str, list[int] | list[str]] = {}
    for col, dtype in fixations.schema.items():
        if dtype == pl.String:
            dummy_fixation_dict[col] = [""]
        else:
            dummy_fixation_dict[col] = [0]
    dummy_fixation = pl.DataFrame(
        dummy_fixation_dict,
        schema=fixations.schema,
    )
    fixations = pl.concat([fixations, dummy_fixation])

    # Adjust AOI indices (fix off by one error).
    aois = aois.with_columns(
        (pl.col(word_index_column) - 1).alias(word_index_column),
    )

    # Get original words of the text and their indices.
    word_indices = aois[word_index_column].to_list()
    words = aois[word_column].to_list()

    # Initialize dictionary for reading measures per word.
    rm_dict = {
        word_index: {
            "word": word,
            "word_index": word_index,
            "FFD": 0,
            "SFD": 0,
            "FD": 0,
            "FPRT": 0,
            "FPFC": 0,
            "FRT": 0,
            "TFT": 0,
            "RRT": 0,
            "RPD_inc": 0,
            "RPD_exc": 0,
            "RBRT": 0,
            "Fix": 0,
            "FPF": 0,
            "RR": 0,
            "FPReg": 0,
            "TRC_out": 0,
            "TRC_in": 0,
            "SL_in": 0,
            "SL_out": 0,
            "TFC": 0,
            "LP": None,
        }
        for word_index, word in zip(word_indices, words)
    }

    # Add a catch-all entry for the dummy fixation and invalid AOIs
    rm_dict[-1] = {
        "word": None,
        "word_index": -1,
        "FFD": 0,
        "SFD": 0,
        "FD": 0,
        "FPRT": 0,
        "FPFC": 0,
        "FRT": 0,
        "TFT": 0,
        "RRT": 0,
        "RPD_inc": 0,
        "RPD_exc": 0,
        "RBRT": 0,
        "Fix": 0,
        "FPF": 0,
        "RR": 0,
        "FPReg": 0,
        "TRC_out": 0,
        "TRC_in": 0,
        "SL_in": 0,
        "SL_out": 0,
        "TFC": 0,
        "LP": None,
    }

    # Variables to track fixation progress.
    right_most_word, cur_fix_word_idx, next_fix_word_idx, next_fix_dur = -1, -1, -1, -1

    # Iterate over fixation data.
    for fixation in fixations.to_dicts():
        try:
            aoi = int(fixation[word_index_column]) - 1
            if aoi not in rm_dict:
                continue
        except (ValueError, TypeError):
            continue

        # Update variables.
        last_fix_word_idx = cur_fix_word_idx
        cur_fix_word_idx = next_fix_word_idx
        cur_fix_dur = next_fix_dur
        if cur_fix_dur is None:
            continue

        next_fix_word_idx = aoi
        next_fix_dur = fixation["duration"]

        if next_fix_dur == 0 and not next_fix_word_idx == -1:
            next_fix_word_idx = cur_fix_word_idx

        right_most_word = max(right_most_word, cur_fix_word_idx)

        if cur_fix_word_idx == -1:
            continue

        # Update reading measures for the current word.
        rm_dict[cur_fix_word_idx]["TFT"] += int(cur_fix_dur)
        rm_dict[cur_fix_word_idx]["TFC"] += 1
        if rm_dict[cur_fix_word_idx]["FD"] == 0:
            rm_dict[cur_fix_word_idx]["FD"] += int(cur_fix_dur)
            rm_dict[cur_fix_word_idx]["LP"] = int(fixation[char_index_column])

        if right_most_word == cur_fix_word_idx:
            if rm_dict[cur_fix_word_idx]["TRC_out"] == 0:
                rm_dict[cur_fix_word_idx]["FPRT"] += int(cur_fix_dur)
                rm_dict[cur_fix_word_idx]["FPFC"] += 1
                if last_fix_word_idx < cur_fix_word_idx:
                    rm_dict[cur_fix_word_idx]["FFD"] += int(cur_fix_dur)
        else:
            rm_dict[right_most_word]["RPD_exc"] += int(cur_fix_dur)

        if cur_fix_word_idx < last_fix_word_idx:
            rm_dict[cur_fix_word_idx]["TRC_in"] += 1
        if cur_fix_word_idx > next_fix_word_idx:
            rm_dict[cur_fix_word_idx]["TRC_out"] += 1
        if cur_fix_word_idx == right_most_word:
            rm_dict[cur_fix_word_idx]["RBRT"] += int(cur_fix_dur)
        if rm_dict[cur_fix_word_idx]["FRT"] == 0 and (
            not next_fix_word_idx == cur_fix_word_idx or next_fix_dur == 0
        ):
            rm_dict[cur_fix_word_idx]["FRT"] = rm_dict[cur_fix_word_idx]["TFT"]
        if rm_dict[cur_fix_word_idx]["SL_in"] == 0:
            rm_dict[cur_fix_word_idx]["SL_in"] = cur_fix_word_idx - last_fix_word_idx
        if rm_dict[cur_fix_word_idx]["SL_out"] == 0:
            rm_dict[cur_fix_word_idx]["SL_out"] = next_fix_word_idx - cur_fix_word_idx

    # Finalize reading measures.
    rm_list = []
    for aoi_key, aoi_rm in sorted(rm_dict.items()):
        if aoi_key == -1:
            continue
        if aoi_rm["FFD"] == aoi_rm["FPRT"]:
            aoi_rm["SFD"] = aoi_rm["FFD"]
        aoi_rm["RRT"] = aoi_rm["TFT"] - aoi_rm["FPRT"]
        aoi_rm["FPF"] = int(aoi_rm["FFD"] > 0)
        aoi_rm["RR"] = int(aoi_rm["RRT"] > 0)
        aoi_rm["FPReg"] = int(aoi_rm["RPD_exc"] > 0)
        aoi_rm["Fix"] = int(aoi_rm["TFT"] > 0)
        aoi_rm["RPD_inc"] = aoi_rm["RPD_exc"] + aoi_rm["RBRT"]

        rm_list.append(aoi_rm)

    return pl.DataFrame(rm_list)

In [ ]:
stimuli = sess.stimuli
words_only_all_trials = []
for stim in stimuli:
    aois = stim.text_stimulus.aois
    words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
    words_only = words_only.with_columns(pl.lit(stim.name).alias("stimulus"))
    words_only_all_trials.append(words_only)

words_df = pl.concat(words_only_all_trials)

In [ ]:
group_columns = [settings.TRIAL_COL, settings.STIMULUS_COL, settings.PAGE_COL]
only_fix = (
    gaze.events.frame.filter(
        (pl.col("name") == settings.FIXATION)
        & (pl.col(settings.WORD_IDX_COL).is_not_null())
    )
    .with_row_count("fixation_id")
    .sort(group_columns + ["onset"])
)

In [ ]:
only_fix

In [ ]:
rm_all_trials = []

# Iterate over pages
for (trial_idx, stim_name, page_idx), fix_df in only_fix.group_by(group_columns):
    page_words = words_df.filter(
        (pl.col(settings.TRIAL_COL) == trial_idx)
        & (pl.col(settings.PAGE_COL) == page_idx)
    )
    rm = compute_reading_measures(
        fixations=fix_df,
        aois=page_words,
        word_index_column=settings.WORD_IDX_COL,
        # to be replaced when words change to unit of analysis
        word_column="word",
    )
    rm = rm.with_columns(
        pl.lit(trial_idx).alias(settings.TRIAL_COL),
        pl.lit(page_idx).alias(settings.PAGE_COL),
        pl.lit(stim_name).alias(settings.STIMULUS_COL),
    )
    rm_all_trials.append(rm)

rm_df = pl.concat(rm_all_trials)
# rename word index to original column name
rm_df = rm_df.rename({"word_index": settings.WORD_IDX_COL})

# adjust reading measures to original format of preprocessing pipeline
rm_df = rm_df.with_columns((1 - pl.col("Fix")).alias("skipped"))

rm_df.drop("Fix")

In [ ]:
preprocessing.save_reading_measures(sid, rm_df.drop("Fix"))

## Stage 4: Comprehension Question Answers
In addition to gaze data, each session contains answers to comprehension questions.
These are extracted from the ASC messages. The answers are matched to the stimulus order using the `question_order_versions.csv` file in the session's logfiles folder.

In [ ]:
answers_csv = sid.answers_dir / f"{sid}_answers.csv"
question_order_csv = (
    sess.session_folder_path / "logfiles" / "question_order_versions.csv"
)
parsed_answers = preprocessing.parse_answers_from_messages(gaze.messages)
source = "asc"

In [ ]:
parsed_answers

In [ ]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

## Final Steps

In the very end, we can create the session and dataset overview and store them as well. In addition, the participant data can be parsed and stored.

For the MultiplEYE data, there is also the option to create a sanity check report.

In [ ]:
multipleye.create_sanity_check_report(
    gaze,
    sess.session_identifier,
    output_dir=settings.OUTPUT_DIR,
    plotting=True,
    overwrite=True,
)

In [ ]:
multipleye.create_session_overview(sess.session_identifier, path=settings.OUTPUT_DIR)
multipleye.create_dataset_overview(path=settings.OUTPUT_DIR)
multipleye.parse_participant_data(settings.OUTPUT_DIR / "participant_data.csv")

In [ ]:
from preprocessing.psychometric_tests.preprocess_psychometric_tests import (
    preprocess_all_sessions,
)

preprocess_all_sessions()